# Data Preparation — Previsão do Vencedor de Confrontos Profissionais de CS2

**Projeto:** Previsão do vencedor de confrontos profissionais de CS2
**Dataset:** [Counter-Strike Pro Matches (Kaggle, ektarr)](https://www.kaggle.com/datasets/ektarr/counter-strike-pro-matches)
**Base:** Machine Learning Canvas do projeto + notebook de Data Understanding

Este notebook já usa os **nomes reais das colunas** do dataset (confirmados a partir do próprio ficheiro), em vez de deteção automática.

### Estrutura real do dataset (confirmada)
- Granularidade: uma linha por **jogo/mapa** (`game_id`), agrupado por confronto (`match_id`) — um confronto BO3/BO5 tem várias linhas (uma por mapa).
- `team1_win`: já vem pronto como variável alvo (1 se a equipa 1 venceu aquele mapa, 0 caso contrário).
- `map_name` / `map_id_*.0`: mapa jogado (já com one-hot encoding parcial feito pelo autor do dataset).
- `bestOf_1.0` / `bestOf_3.0` / `bestOf_5.0`: formato do confronto (já one-hot).
- **Não existe coluna de ranking** — ao contrário do que o Canvas assumia. Vamos construir um **rating Elo dinâmico** como proxy,
  calculado apenas com o histórico de jogos anteriores.
- Dezenas de colunas de estatísticas **pós-jogo** por jogador (`kills`, `deaths`, `assists`, `adr`, `kast`, `kddiff`) e placares
  (`score1_game`, `score2_game`, `score1_match`, `score2_match`, `is_total`, `games_played`) — estas são conhecidas **depois**
  do jogo terminar, logo constituem **leakage** e são excluídas das features de pré-confronto.

Objetivos deste notebook:
1. Carregar os dados.
2. Validar as hipóteses sobre a estrutura (`is_total`, nº de mapas por confronto).
3. Limpar os dados.
4. Construir features pré-confronto sem leakage: Elo (proxy de ranking), taxa histórica de vitórias, forma recente, h2h, mapa, formato.
5. Fazer split treino/teste temporal.
6. Guardar o dataset preparado.

## 1. Setup

In [23]:
!pip install -q kagglehub pandas numpy scikit-learn

import kagglehub
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42


## 2. Carregamento dos dados

In [24]:
dataset_path = kagglehub.dataset_download("ektarr/counter-strike-pro-matches")
print("Dataset descarregado para:", dataset_path)

dataframes = {}
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith('.csv'):
            full_path = os.path.join(root, f)
            dataframes[f] = pd.read_csv(full_path)
            print(f"Carregado: {f} -> shape {dataframes[f].shape}")

if len(dataframes) == 1:
    df = list(dataframes.values())[0]
else:
    nome_principal = max(dataframes, key=lambda k: dataframes[k].shape[0])
    df = dataframes[nome_principal]
    print(f"\nMúltiplos ficheiros encontrados. A usar '{nome_principal}'.")

print(df.shape)
df.head()


Using Colab cache for faster access to the 'counter-strike-pro-matches' dataset.
Dataset descarregado para: /kaggle/input/counter-strike-pro-matches
Carregado: players.csv -> shape (1398, 2)
Carregado: teams.csv -> shape (19846, 2)
Carregado: tournaments.csv -> shape (344, 1)
Carregado: cs2_tier1_games.csv -> shape (11151, 98)
Carregado: cs2_tier3_games.csv -> shape (1220, 98)
Carregado: cs2_all_tiers_games.csv -> shape (20676, 98)
Carregado: cs2_tier2_games.csv -> shape (8305, 98)

Múltiplos ficheiros encontrados. A usar 'cs2_all_tiers_games.csv'.
(20676, 98)


,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,is_total,bestOf,score1_game,score2_game,map_id,map_name,datetime,team1_win,games_played,team1_player1_id,team1_player2_id,team1_player3_id,team1_player4_id,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id,team1_player1,team1_player2,team1_player3,team1_player4,team1_player5,team2_player1,team2_player2,team2_player3,team2_player4,team2_player5,team1_player1_kills,team1_player1_deaths,team1_player1_assists,team1_player1_adr,team1_player1_kast,team1_player1_kddiff,team1_player2_kills,team1_player2_deaths,team1_player2_assists,team1_player2_adr,team1_player2_kast,team1_player2_kddiff,team1_player3_kills,team1_player3_deaths,team1_player3_assists,team1_player3_adr,team1_player3_kast,team1_player3_kddiff,team1_player4_kills,team1_player4_deaths,team1_player4_assists,team1_player4_adr,team1_player4_kast,team1_player4_kddiff,team1_player5_kills,team1_player5_deaths,team1_player5_assists,team1_player5_adr,team1_player5_kast,team1_player5_kddiff,team2_player1_kills,team2_player1_deaths,team2_player1_assists,team2_player1_adr,team2_player1_kast,team2_player1_kddiff,team2_player2_kills,team2_player2_deaths,team2_player2_assists,team2_player2_adr,team2_player2_kast,team2_player2_kddiff,team2_player3_kills,team2_player3_deaths,team2_player3_assists,team2_player3_adr,team2_player3_kast,team2_player3_kddiff,team2_player4_kills,team2_player4_deaths,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
0,7303015,154188.0,Super DraculaN Season 1,295838,Inner Circle,295839,Acend,3.0,1.0,False,5.0,13.0,8.0,4.0,Overpass,2026-06-28 20:00:00,1,4.0,2799.0,2109.0,8304.0,26198.0,25560.0,691.0,NaN,1332.0,2774.0,1459.0,"headtr1ck (Daniil Valitov), UA","cptkurtka023 (Arsenly Derevinskly), UA",zeRRoFIX,onic,Dawy,"SPELLAN (Teodor Nikolov), BG",NaN,"KalubeR (Kalin Erenditsov), BG","h4rn (Deyvid Benchev), BG","REDSTAR (Hristiyan Pironkov), BG",16.0,11.0,3.0,76.5,71.4,5.0,17.0,12.0,7.0,107.0,81.0,5.0,11.0,10.0,5.0,59.3,66.7,1.0,12.0,10.0,2.0,54.5,76.2,2.0,9.0,7.0,3.0,51.9,85.7,2.0,10.0,14.0,2.0,56.2,57.1,-4.0,NaN,NaN,NaN,NaN,NaN,NaN,18.0,13.0,5.0,89.3,81.0,5.0,6.0,12.0,5.0,44.5,66.7,-6.0,7.0,13.0,7.0,43.8,76.2,-6.0
1,7303015,154189.0,Super DraculaN Season 1,295838,Inner Circle,295839,Acend,3.0,1.0,False,5.0,13.0,8.0,24.0,Ancient,2026-06-28 20:00:00,1,4.0,2799.0,2109.0,8304.0,26198.0,25560.0,691.0,NaN,1332.0,2774.0,1459.0,"headtr1ck (Daniil Valitov), UA","cptkurtka023 (Arsenly Derevinskly), UA",zeRRoFIX,onic,Dawy,"SPELLAN (Teodor Nikolov), BG",NaN,"KalubeR (Kalin Erenditsov), BG","h4rn (Deyvid Benchev), BG","REDSTAR (Hristiyan Pironkov), BG",14.0,12.0,2.0,65.0,71.4,2.0,14.0,14.0,7.0,77.2,61.9,0.0,13.0,10.0,5.0,78.0,81.0,3.0,16.0,11.0,3.0,74.5,76.2,5.0,16.0,14.0,7.0,78.9,76.2,2.0,8.0,19.0,4.0,57.0,57.1,-11.0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,14.0,6.0,70.4,81.0,-1.0,12.0,11.0,2.0,67.4,81.0,1.0,13.0,16.0,4.0,69.3,66.7,-3.0
2,7303015,-154186.0,Super DraculaN Season 1,295838,Inner Circle,295839,Acend,3.0,1.0,True,5.0,0.0,0.0,23.0,Anubis,2026-06-28 20:00:00,1,4.0,2799.0,2109.0,8304.0,26198.0,25560.0,691.0,NaN,1332.0,2774.0,1459.0,"headtr1ck (Daniil Valitov), UA","cptkurtka023 (Arsenly Derevinskly), UA",zeRRoFIX,onic,Dawy,"SPELLAN (Teodor Nikolov), BG",NaN,"KalubeR (Kalin Erenditsov), BG","h4rn (Deyvid Benchev), BG","REDSTAR (Hristiyan Pironkov), BG",64.0,50.0,15.0,67.8,70.9,14.0,56.0,58.0,21.0,79.8,74.1,-2.0,64.0,50.0,21.0,77.1,72.1,14.0,54.0,49.0,18.0,66.2,74.4,5.0,60.0,51.0,18.0,74.4,77.7,9.0,40.0,69.0,22.0,62.1,59.4,-29.0,NaN,NaN,NaN,NaN,NaN,NaN,63.0,61.0,19.0,77.7,75.5,2.0,46.0,48.0,16.0,58.3,74.1,-2.0,57.0,65.0,18.0,73.2,69.7,-8.0
3,7303015,154187.0,Super DraculaN Season 1,295838,Inner Circle,295839,Acend,3.0,1.0,False,5.0,13.0,10.0,6.0,Dust2,2026-06-28 20:00:00,1,4.0,2799.0,2109.0,8304.0

## 3. Validar hipóteses sobre a estrutura dos dados

Antes de decidir a granularidade a usar, confirmar o significado de `is_total` e `games_played`:
- Se `is_total` marcar uma linha-resumo do confronto inteiro (para além das linhas por mapa), essas linhas devem ser **excluídas**
  da granularidade "por mapa" para não haver dupla contagem.
- Confirmar quantos `game_id` existem tipicamente por `match_id` (deve bater certo com o formato: 1, até 3, ou até 5).

In [25]:
print("Valores únicos de is_total:")
print(df['is_total'].value_counts(dropna=False))

print("\nValores únicos de games_played:")
print(df['games_played'].value_counts(dropna=False).sort_index())

print("\nNº de linhas (jogos) por match_id — distribuição:")
print(df.groupby('match_id').size().value_counts().sort_index())

print("\nExemplo de um match_id com várias linhas:")
exemplo_match = df['match_id'].value_counts().index[0]
display(df[df['match_id'] == exemplo_match][
    ['match_id', 'game_id', 'team1', 'team2', 'map_name', 'is_total',
     'score1_game', 'score2_game', 'score1_match', 'score2_match', 'team1_win']
])


Valores únicos de is_total:
is_total
False    10753
True      9923
Name: count, dtype: int64

Valores únicos de games_played:
games_played
1.0    2743
2.0    9013
3.0    8691
4.0     132
5.0      96
NaN       1
Name: count, dtype: int64

Nº de linhas (jogos) por match_id — distribuição:
1    4750
2    1202
3    2406
4    1530
5      26
6       9
Name: count, dtype: int64

Exemplo de um match_id com várias linhas:


,match_id,game_id,team1,team2,map_name,is_total,score1_game,score2_game,score1_match,score2_match,team1_win
7879,2691788,122900.0,Team Vitality,Team Falcons,Inferno,False,9.0,13.0,3.0,2.0,0
7880,2691788,-122903.0,Team Vitality,Team Falcons,NaN,True,NaN,NaN,3.0,2.0,0
7881,2691788,122902.0,Team Vitality,Team Falcons,Train,False,13.0,6.0,3.0,2.0,1
7882,2691788,122901.0,Team Vitality,Team Falcons,Dust2,False,13.0,6.0,3.0,2.0,1
7883,2691788,122904.0,Team Vitality,Team Falcons,Nuke,False,13.0,8.0,3.0,2.0,1
7884,2691788,122903.0,Team Vitality,Team Falcons,Mirage,False,14.0,16.0,3.0,2.0,0


In [26]:
# AJUSTAR conforme o resultado da célula acima:
# Se is_total==True/1 identificar linhas-resumo do confronto (sem mapa específico), remover essas linhas
# e manter apenas a granularidade "por mapa" (is_total == False/0), que é a que tem team1_win e map_name válidos por jogo.

antes = len(df)
if df['is_total'].dtype == bool:
    df = df[df['is_total'] == False].copy()
else:
    df = df[df['is_total'] == 0].copy()
print(f"Linhas removidas por serem resumo de confronto (is_total): {antes - len(df)}")
print(f"Linhas restantes (nível de mapa/jogo): {len(df)}")


Linhas removidas por serem resumo de confronto (is_total): 9923
Linhas restantes (nível de mapa/jogo): 10753


## 4. Limpeza dos dados

- Converter `datetime` e ordenar cronologicamente (essencial para Elo, taxa histórica, forma recente e h2h, que só podem olhar
  para o **passado** de cada jogo).
- Remover duplicados e linhas sem informação essencial (equipas ou `team1_win` em falta).

In [27]:
print(f"Linhas antes da limpeza: {len(df)}")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Após remover duplicados: {len(df)}")

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
n_sem_data = df['datetime'].isna().sum()
print(f"Linhas sem data válida (removidas): {n_sem_data}")
df = df.dropna(subset=['datetime']).sort_values(['datetime', 'match_id', 'game_id']).reset_index(drop=True)

colunas_essenciais = ['team1', 'team2', 'team1_win']
antes = len(df)
df = df.dropna(subset=colunas_essenciais).reset_index(drop=True)
print(f"Linhas removidas por falta de equipas/team1_win: {antes - len(df)}")

df['target'] = df['team1_win'].astype(int)
print(f"\nLinhas finais: {len(df)}")
print(df['target'].value_counts(normalize=True).rename('proporção'))


Linhas antes da limpeza: 10753
Após remover duplicados: 10753
Linhas sem data válida (removidas): 0
Linhas removidas por falta de equipas/team1_win: 0

Linhas finais: 10753
target
1    0.536036
0    0.463964
Name: proporção, dtype: float64


## 5. Feature engineering (features pré-jogo, sem leakage)

Todas as features abaixo usam **apenas jogos anteriores** a cada linha (a mesma equipa, ordenada por `datetime`), nunca
informação do próprio jogo que se quer prever. Isto evita o leakage que o Canvas alerta ser um risco crítico.

### 5.1 Elo rating (proxy de ranking)

Como o dataset não tem uma coluna de ranking, calculamos um rating Elo dinâmico por equipa: começa em 1500 para todas as equipas
e é atualizado após cada jogo. A feature usada é o Elo **antes** do jogo em questão (nunca depois).

In [28]:
def calcular_elo(df, col_team1='team1', col_team2='team2', col_target='target', k=32, elo_inicial=1500):
    elo = {}
    elo_team1_antes = []
    elo_team2_antes = []

    for _, row in df.iterrows():
        t1, t2, alvo = row[col_team1], row[col_team2], row[col_target]

        r1 = elo.get(t1, elo_inicial)
        r2 = elo.get(t2, elo_inicial)

        elo_team1_antes.append(r1)
        elo_team2_antes.append(r2)

        esperado_t1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
        resultado_t1 = 1 if alvo == 1 else 0

        elo[t1] = r1 + k * (resultado_t1 - esperado_t1)
        elo[t2] = r2 + k * ((1 - resultado_t1) - (1 - esperado_t1))

    return elo_team1_antes, elo_team2_antes

df['elo_team1'], df['elo_team2'] = calcular_elo(df)
df['elo_diff'] = df['elo_team1'] - df['elo_team2']
df[['team1', 'team2', 'elo_team1', 'elo_team2', 'elo_diff']].head()


,team1,team2,elo_team1,elo_team2,elo_diff
0,Preasy Esport,FORZE,1500.000000,1500.000000,0.000000
1,Pompa Team,RUBIK,1500.000000,1500.000000,0.000000
2,ORKS,IKLA,1500.000000,1500.000000,0.000000
3,ORKS,IKLA,1484.000000,1516.000000,-32.000000
4,ORKS,IKLA,1501.469502,1498.530498,2.939003


### 5.2 Taxa histórica de vitórias e forma recente (últimos N jogos)

In [29]:
N_RECENTE = 5

def taxa_vitorias_e_forma(df, col_team1='team1', col_team2='team2', col_target='target', n_recente=N_RECENTE):
    historico = {}
    win_rate_t1, win_rate_t2 = [], []
    forma_t1, forma_t2 = [], []

    for _, row in df.iterrows():
        t1, t2, alvo = row[col_team1], row[col_team2], row[col_target]

        hist_t1 = historico.get(t1, [])
        hist_t2 = historico.get(t2, [])

        win_rate_t1.append(np.mean(hist_t1) if hist_t1 else np.nan)
        win_rate_t2.append(np.mean(hist_t2) if hist_t2 else np.nan)
        forma_t1.append(np.mean(hist_t1[-n_recente:]) if hist_t1 else np.nan)
        forma_t2.append(np.mean(hist_t2[-n_recente:]) if hist_t2 else np.nan)

        historico.setdefault(t1, []).append(1 if alvo == 1 else 0)
        historico.setdefault(t2, []).append(1 if alvo == 0 else 0)

    return win_rate_t1, win_rate_t2, forma_t1, forma_t2

(df['win_rate_team1'], df['win_rate_team2'],
 df['forma_recente_team1'], df['forma_recente_team2']) = taxa_vitorias_e_forma(df)

df['win_rate_diff'] = df['win_rate_team1'] - df['win_rate_team2']
df['forma_recente_diff'] = df['forma_recente_team1'] - df['forma_recente_team2']


### 5.3 Confrontos diretos anteriores (h2h)

In [30]:
def h2h_historico(df, col_team1='team1', col_team2='team2', col_target='target'):
    historico_pares = {}
    h2h_t1, h2h_t2 = [], []

    for _, row in df.iterrows():
        t1, t2, alvo = row[col_team1], row[col_team2], row[col_target]
        par = frozenset([t1, t2])

        vit_t1, vit_t2 = historico_pares.get(par, (0, 0))
        h2h_t1.append(vit_t1)
        h2h_t2.append(vit_t2)

        if alvo == 1:
            vit_t1 += 1
        else:
            vit_t2 += 1
        historico_pares[par] = (vit_t1, vit_t2)

    return h2h_t1, h2h_t2

df['h2h_wins_team1'], df['h2h_wins_team2'] = h2h_historico(df)
df['h2h_diff'] = df['h2h_wins_team1'] - df['h2h_wins_team2']


### 5.4 Mapa e formato

Já vêm codificados no dataset original (`map_id_*.0`, `bestOf_1.0/3.0/5.0`) — não é preciso one-hot encoding adicional.
`map_name` é mantido apenas como referência legível, não como feature do modelo (redundante com `map_id_*`).

In [31]:
colunas_map = [c for c in df.columns if c.startswith('map_id_')]
colunas_bestof = [c for c in df.columns if c.startswith('bestOf_')]
print("Colunas de mapa (one-hot):", colunas_map)
print("Colunas de formato (one-hot):", colunas_bestof)


Colunas de mapa (one-hot): []
Colunas de formato (one-hot): []


## 6. Exclusão explícita de colunas com leakage

Estatísticas de desempenho por jogador (`kills`, `deaths`, `assists`, `adr`, `kast`, `kddiff`) e placares (`score1_game`,
`score2_game`, `score1_match`, `score2_match`, `games_played`) só são conhecidos **depois** do jogo — não podem ser usados
como features de pré-jogo. IDs/nomes de jogadores também ficam de fora nesta versão (fora do âmbito das features do Canvas).

In [32]:
colunas_leakage = [c for c in df.columns if any(s in c for s in [
    '_kills', '_deaths', '_assists', '_adr', '_kast', '_kddiff',
    'score1_game', 'score2_game', 'score1_match', 'score2_match', 'games_played'
])]
colunas_jogadores_id = [c for c in df.columns if 'player' in c]

print(f"Colunas de leakage identificadas e excluídas: {len(colunas_leakage)}")
print(f"Colunas de jogadores excluídas (fora do âmbito do Canvas): {len(colunas_jogadores_id)}")


Colunas de leakage identificadas e excluídas: 65
Colunas de jogadores excluídas (fora do âmbito do Canvas): 80


## 7. Seleção do conjunto final de features

In [33]:
colunas_pre_jogo = [
    'elo_diff', 'elo_team1', 'elo_team2',
    'win_rate_diff', 'win_rate_team1', 'win_rate_team2',
    'forma_recente_diff', 'forma_recente_team1', 'forma_recente_team2',
    'h2h_diff', 'h2h_wins_team1', 'h2h_wins_team2',
] + colunas_map + colunas_bestof

colunas_finais = colunas_pre_jogo + ['datetime', 'target']
colunas_finais = [c for c in colunas_finais if c in df.columns]

df_prep = df[colunas_finais].copy()
print(f"Shape final do dataset preparado: {df_prep.shape}")
df_prep.head()


Shape final do dataset preparado: (10753, 14)


,elo_diff,elo_team1,elo_team2,win_rate_diff,win_rate_team1,win_rate_team2,forma_recente_diff,forma_recente_team1,forma_recente_team2,h2h_diff,h2h_wins_team1,h2h_wins_team2,datetime,target
0,0.000000,1500.000000,1500.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2023-10-25 18:00:00,1
1,0.000000,1500.000000,1500.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2023-10-25 20:00:00,1
2,0.000000,1500.000000,1500.000000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2023-10-25 20:30:00,0
3,-32.000000,1484.000000,1516.000000,-1.0,0.0,1.0,-1.0,0.0,1.0,-1,0,1,2023-10-25 20:30:00,1
4,2.939003,1501.469502,1498.530498,0.0,0.5,0.5,0.0,0.5,0.5,0,1,1,2023-10-25 20:30:00,0


## 8. Tratamento de valores em falta

Jogos iniciais de uma equipa (sem histórico anterior) geram `NaN` em `win_rate_*` e `forma_recente_*`. Preenchemos com 0.5
(taxa neutra) — o Elo já começa com um valor neutro (1500) por definição, não precisa de tratamento adicional.

In [34]:
cols_taxa = [c for c in df_prep.columns if 'win_rate' in c or 'forma_recente' in c]
for c in cols_taxa:
    n_nan = df_prep[c].isna().sum()
    if n_nan:
        print(f"{c}: {n_nan} valores em falta -> preenchidos com 0.5")
        df_prep[c] = df_prep[c].fillna(0.5)

print("\nValores em falta remanescentes:")
print(df_prep.isna().sum()[df_prep.isna().sum() > 0])


win_rate_diff: 338 valores em falta -> preenchidos com 0.5
win_rate_team1: 154 valores em falta -> preenchidos com 0.5
win_rate_team2: 243 valores em falta -> preenchidos com 0.5
forma_recente_diff: 338 valores em falta -> preenchidos com 0.5
forma_recente_team1: 154 valores em falta -> preenchidos com 0.5
forma_recente_team2: 243 valores em falta -> preenchidos com 0.5

Valores em falta remanescentes:
Series([], dtype: int64)


## 9. Divisão treino/teste temporal

Split cronológico (80/20): treino nos jogos mais antigos, teste nos mais recentes — evita treinar com informação "do futuro".

In [35]:
df_prep = df_prep.sort_values('datetime').reset_index(drop=True)
corte = int(len(df_prep) * 0.8)

train = df_prep.iloc[:corte].drop(columns=['datetime'])
test = df_prep.iloc[corte:].drop(columns=['datetime'])

print(f"Treino: {train.shape[0]} jogos ({df_prep['datetime'].iloc[0].date()} a {df_prep['datetime'].iloc[corte-1].date()})")
print(f"Teste:  {test.shape[0]} jogos ({df_prep['datetime'].iloc[corte].date()} a {df_prep['datetime'].iloc[-1].date()})")

X_train, y_train = train.drop(columns=['target']), train['target']
X_test, y_test = test.drop(columns=['target']), test['target']


Treino: 8602 jogos (2023-10-25 a 2025-11-05)
Teste:  2151 jogos (2025-11-05 a 2026-06-28)


## 10. Guardar o dataset preparado

In [36]:
train.to_csv('cs2_matches_train.csv', index=False)
test.to_csv('cs2_matches_test.csv', index=False)

print("Ficheiros guardados: cs2_matches_train.csv, cs2_matches_test.csv")
print(f"\n{X_train.shape[1]} features finais:")
print(list(X_train.columns))


Ficheiros guardados: cs2_matches_train.csv, cs2_matches_test.csv

12 features finais:
['elo_diff', 'elo_team1', 'elo_team2', 'win_rate_diff', 'win_rate_team1', 'win_rate_team2', 'forma_recente_diff', 'forma_recente_team1', 'forma_recente_team2', 'h2h_diff', 'h2h_wins_team1', 'h2h_wins_team2']


## 11. Síntese da preparação

- **Granularidade:** por mapa/jogo (`game_id`), após remover linhas de resumo de confronto (`is_total`) — validar se esta
  hipótese estava correta ao correr a secção 3.
- **Alvo:** `target` = `team1_win` (1 = equipa 1 venceu o mapa, 0 = equipa 2 venceu).
- **Ranking:** não existia no dataset original — substituído por um **rating Elo dinâmico**, calculado apenas com o passado.
- **Outras features:** taxa histórica de vitórias, forma recente (últimos 5 jogos), confrontos diretos (h2h), mapa e formato
  (já one-hot no dataset original).
- **Leakage excluído:** estatísticas de jogador pós-jogo e todos os placares.
- **Split:** cronológico (80/20).

**Pontos a validar depois de correr no Colab:**
- Confirmar na secção 3 se `is_total` foi interpretado corretamente (ver o exemplo impresso de um `match_id` com várias linhas).
- Se o objetivo for prever o vencedor do **confronto completo** (BO3/BO5) e não de cada mapa individual, é necessário agregar
  as previsões por `match_id` numa fase posterior (ex. equipa que ganhar mais mapas previstos vence o confronto) — este notebook
  prepara os dados ao nível do mapa, que é a granularidade nativa do dataset.
- Validar se o Elo com `k=32` e valor inicial 1500 é adequado, ou se deve ser calibrado/testado como hiperparâmetro.
- Ponderar se `tournament` deveria entrar como feature (não incluído nesta versão, por simplicidade e por ter alta cardinalidade).